In [ ]:

!apt-get update -qq
!apt-get install -y zstd
%pip -q install -U langchain-core langchain-ollama pydantic
!curl -fsSL https://ollama.com/install.sh | sh


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 39 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, urllib.request

ollama_process = None
try:
    urllib.request.urlopen('http://localhost:11434', timeout=2)
    print('Ollama уже запущена.')
except Exception:
    ollama_process = subprocess.Popen(
        ['ollama', 'serve'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    for _ in range(30):
        try:
            urllib.request.urlopen('http://localhost:11434', timeout=2)
            print('Ollama запущена.')
            break
        except Exception:
            time.sleep(2)
    else:
        raise RuntimeError('Не удалось запустить Ollama. Перезапусти среду Colab и повтори.')

!ollama pull llama3.2

Ollama уже запущена.



In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/langchain_memory_colab')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
MEMORY_FILE = PROJECT_DIR / 'memory.json'
print('Файл памяти:', MEMORY_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Файл памяти: /content/drive/MyDrive/langchain_memory_colab/memory.json


In [ ]:

import subprocess
import time
import urllib.request

# Запускаем сервер Ollama
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

# Ждём, пока сервер начнёт отвечать
for attempt in range(30):
    try:
        response = urllib.request.urlopen(
            "http://127.0.0.1:11434",
            timeout=2
        )
        print("Ollama работает:", response.read().decode())
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError(
        "Ollama не запустилась. Проверь следующую ячейку диагностики."
    )


Ollama работает: Ollama is running


In [ ]:
!ollama list

NAME               ID              SIZE      MODIFIED     
llama3.2:latest    a80c4f17acd5    2.0 GB    1 second ago    


In [ ]:
import json
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

MODEL_NAME = 'llama3.2'
USER_ID = 'student_1'

llm = ChatOllama(
    model=MODEL_NAME,
    temperature=0,
    num_predict=256,
    client_kwargs={'timeout': 180}
)

class UserFacts(BaseModel):
    name: str | None = Field(default=None, description='Имя пользователя')
    skills: list[str] = Field(default_factory=list, description='Навыки пользователя')
    level: str | None = Field(default=None, description='Уровень знаний')
    goals: list[str] = Field(default_factory=list, description='Цели пользователя')

print('Модель и структура фактов готовы.')

Модель и структура фактов готовы.


In [ ]:
def load_memory():
    if not MEMORY_FILE.exists():
        return {}
    try:
        with open(MEMORY_FILE, 'r', encoding='utf-8') as file:
            return json.load(file)
    except (json.JSONDecodeError, OSError):
        print('Предупреждение: файл памяти не прочитан; будет создана новая.')
        return {}

def save_memory(data):
    with open(MEMORY_FILE, 'w', encoding='utf-8') as file:
        json.dump(data, file, ensure_ascii=False, indent=4)

class FileMemory:
    def __init__(self, user_id):
        self.user_id = user_id
        self.data = load_memory()
        if user_id not in self.data:
            self.data[user_id] = UserFacts().model_dump()
            save_memory(self.data)

    def load_memory_variables(self):
        return self.data[self.user_id].copy()

    def save_context(self, new_facts):
        current = self.data[self.user_id]
        for key in ('name', 'level'):
            value = new_facts.get(key)
            if value:
                current[key] = value
        for key in ('skills', 'goals'):
            for item in new_facts.get(key, []):
                item = item.strip()
                if item and item not in current[key]:
                    current[key].append(item)
        save_memory(self.data)

memory = FileMemory(USER_ID)
print('Файловая память готова.')

Файловая память готова.


In [ ]:
extract_prompt = ChatPromptTemplate.from_messages([
    ('system', '''Извлеки только факты о самом пользователе, явно сообщённые в текущем сообщении.
Верни JSON-объект с полями:
- name: строка или null
- skills: список строк
- level: строка или null
- goals: список строк
Не выдумывай факты. Если факт не указан, используй null для name и level, пустой список для skills и goals.
Сохраняй формулировки на русском, если пользователь говорит по-русски.
Не считай вопросы о других людях и гипотетические примеры фактами о пользователе.'''),
    ('human', '{message}')
])

extract_chain = extract_prompt | llm.with_structured_output(
    UserFacts,
    method='json_mode'
)

def extract_facts(message):
    return extract_chain.invoke({'message': message}).model_dump()

# Тест при необходимости:
# print(extract_facts('Меня зовут Алексей, я изучаю Python.'))

In [ ]:
import json

def chat(user_message):
    # Загружаем сохранённую память
    with open(MEMORY_FILE, "r", encoding="utf-8") as f:
        all_memory = json.load(f)

    profile = all_memory.get("student_1", {})

    # Формируем контекст из фактов пользователя
    context = f"""
    Имя: {profile.get('name', 'неизвестно')}
    Навыки: {', '.join(profile.get('skills', []))}
    Уровень: {profile.get('level') or 'не указан'}
    Цели: {', '.join(profile.get('goals', []))}
    """

    prompt = f"""
    Ты — дружелюбный учебный помощник.
    Используй профиль пользователя для ответа.

    Профиль:
    {context}

    Сообщение пользователя: {user_message}

    Ответь кратко на русском языке.
    """

    response = test_llm.invoke(prompt)
    return response.content

In [ ]:
user_message = 'Меня зовут Алексей, я изучаю Python и хочу стать аналитиком данных.'
print('Ты:', user_message)
print('\nАгент:', chat(user_message))

Ты: Меня зовут Алексей, я изучаю Python и хочу стать аналитиком данных.

Агент: Приветствую тебя, Алексей! Пleased to meet you! 

Мы можем начать с того, что Python — это отличный язык для анализа данных. Если у тебя есть базовый опыт Python, мы можем начать с простых задач, таких как обработка CSV-файлов или работа с библиотекой Pandas.




In [ ]:
!cat /tmp/ollama.log

time=2026-10-14T13:21:40.235Z level=INFO source=routes.go:2439 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_IGPU_ENABLE: OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MAX_TRANSFER_STREAMS:4 OLLAMA_MODELS:/root/.ollama/models OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NO_CLOUD:false OLLAMA_NUM_PARALLEL:1 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https:

In [ ]:
show_memory()

--- СОХРАНЁННАЯ ПАМЯТЬ ---
{
    "name": "Алексей",
    "skills": [
        "Python",
        "Аналитика данных"
    ],
    "level": null,
    "goals": [
        "Стать квалифицированным аналитиком данных"
    ]
}
Файл: /content/drive/MyDrive/langchain_memory_colab/memory.json
--------------------------


In [ ]:
print(chat("Как меня зовут и какая у меня цель?"))

Приветствую Алексея!

Меня зовут Бот, и я здесь для того, чтобы помочь тебе в твоем пути к квалификации аналитика данных. Как я уже понял, твоей целью является получение квалификации в этой области, и я готов помочь тебе в этом. Как я могу assisting thee?
